# Lab 0 — Prerequisites & Foundations

**What this lab is.** This is the foundation lab. Before we can build any of CareConnect's
"agents", we need a few shared building blocks in place: the libraries installed, our
identity and region confirmed, a searchable **Knowledge Base** built from Riverside Health's
approved documents, and a **safety Guardrail** that blocks medical advice. This lab creates
all of that.

**Why we do it.** CareConnect answers patient questions *only* from approved hospital
documents, and it must *never* give medical advice. Those two promises need infrastructure:
a Knowledge Base to hold and search the approved documents, and a Guardrail to enforce the
"no diagnosis / no dosage / no treatment / no triage" rules. Everything in later labs builds
on what we create here.

**Why it's needed in this project specifically.** In a hospital setting, a wrong or made-up
answer is dangerous. By grounding every answer in an approved-documents Knowledge Base and
wrapping it in a Guardrail, we make the assistant *safe by design* rather than hoping the
model behaves.

**How it helps the rest of the build.** Every later lab (retrieval, verification, the
supervisor, the API, the frontend) reads the IDs we save here (the Knowledge Base ID, the
Guardrail ID) from AWS Systems Manager **Parameter Store**. So this lab is the source of
truth the whole system points back to.

**The use case.** A patient asks "How do I prepare for my appointment?" — the answer must
come from Riverside Health's own leaflet, with a citation, and anything clinical must be
refused. This lab lays the groundwork that makes that possible.

> **Reminder:** we reuse the existing `careconnect-approved-docs` S3 bucket (your documents),
> and every *new* thing we create is named with a `-sdk` suffix so it never clashes with the
> build you did in the console.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [2]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Install the software libraries

**What:** install the Python packages this project needs (the AWS SDK `boto3`, the
`strands-agents` framework we use to build agents, and the Bedrock AgentCore packages).

**Why:** these libraries are the tools we build with. Without them, none of the later
`import` lines would work. You only need to do this once per kernel session.

In [3]:
# WHAT THIS CELL DOES (plain English):
# - '%pip install -r requirements.txt' reads the list of required libraries from the
#   requirements.txt file and installs them into this notebook's Python environment.
# - You may see a lot of red "dependency conflict" warnings from SageMaker's own
#   pre-installed packages. Those are WARNINGS, not errors — they are safe to ignore here.
# - After it finishes, you may be prompted to restart the kernel; if so, restart and continue.
# Run once per kernel. Restart the kernel afterwards if prompted.
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


### Step 2 — Load our helpers and confirm who/where we are

**What:** import our shared helper module and print the AWS account, region, and the bucket
we're reusing.

**Why:** this confirms the notebook is talking to the *right* AWS account and the *right*
region (us-east-1). Getting the region wrong is a common cause of "resource not found"
errors later, so we check it up front.

In [4]:
# WHAT THIS CELL DOES (plain English):
# - 'import lab_helpers.utils as u' loads our shared settings/helpers under the short name 'u'.
# - The print lines show which AWS account and region we're using, and which documents bucket
#   we're reusing, so you can visually confirm everything points where you expect.
# - The 'assert' line is a safety check: it STOPS the notebook if the region is not us-east-1,
#   because the rest of the build expects that region.
import boto3, json, time
import lab_helpers.utils as u

print("Account:", u.get_aws_account_id())
print("Region :", u.REGION)
print("Docs bucket (reused):", u.EXISTING_DOCS_BUCKET, "/", u.DOCS_PREFIX)
print("Suffix for new resources:", u.RESOURCE_SUFFIX)
assert u.REGION == "us-east-1", "Switch your notebook Region to us-east-1 to match the existing build."

Account: 831963379350
Region : us-east-1
Docs bucket (reused): careconnect-approved-docs / approved/
Suffix for new resources: -sdk


### Step 3 — Confirm the approved documents are present

**What:** list the files in the existing `careconnect-approved-docs` bucket (we only *look*;
we do not change anything).

**Why:** the Knowledge Base we build next reads from this bucket. If the approved documents
aren't there, there's nothing to search, so we check first and stop early if it's empty.

In [5]:
# WHAT THIS CELL DOES (plain English):
# - Connects to Amazon S3 (AWS file storage) and lists the objects (files) in our
#   approved-documents bucket, under the 'approved/' folder.
# - Prints how many it found and shows the first ten names, so you can confirm the
#   documents are really there.
# - The 'assert docs' line stops the notebook if the bucket is empty.
s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=u.EXISTING_DOCS_BUCKET, Prefix=u.DOCS_PREFIX)
docs = [o["Key"] for o in resp.get("Contents", [])]
print(f"Found {len(docs)} objects under {u.DOCS_PREFIX} (showing first 10):")
for k in docs[:10]:
    print("  ", k)
assert docs, "No approved documents found — check the bucket name in lab_helpers/utils.py"

Found 32 objects under approved/ (showing first 10):
   approved/
   approved/advance-directives.md
   approved/appointment-booking.md
   approved/appointment-preparation-general.md
   approved/appointment-reschedule-cancel.md
   approved/billing-faq.md
   approved/contact-directory.md
   approved/discharge-general-info.md
   approved/emergency-and-urgent-care-info.md
   approved/escalation-policy.md


### Step 4 — Create the Knowledge Base's permission role

**What:** create an IAM **role** — a set of permissions — that the Knowledge Base will
"wear" when it reads your documents and calls the embedding model.

**Why:** in AWS, one service can't touch another unless it's given permission. This role is
the least-privilege permission slip that lets Bedrock read *only* this bucket and use *only*
the embedding model — nothing more.

In [6]:
# WHAT THIS CELL DOES (plain English):
# - Calls our helper to create (or reuse) the IAM role the Knowledge Base needs.
# - Prints the role's ARN (its unique AWS identifier). We'll hand this role to Bedrock next.
kb_role_arn = u.create_kb_execution_role()
print("KB role:", kb_role_arn)

Reusing existing role CareConnectKBRole-sdk
KB role: arn:aws:iam::831963379350:role/CareConnectKBRole-sdk


### Step 5 — Build the Knowledge Base (the searchable memory)

**What:** create a Bedrock **Knowledge Base** backed by **S3 Vectors**. First we create the
vector store (the bucket + index that hold the "searchable maths" version of the documents),
then we create the Knowledge Base that points at it.

**Why:** a Knowledge Base turns plain documents into something the assistant can *search by
meaning*. When a patient asks a question, we find the most relevant approved passages and
answer only from those. **S3 Vectors** is the cheapest option for this (no always-on search
cluster to pay for), which is why we chose it.

**Why two cells:** the AWS API does not auto-create the vector store, so we create the vector
bucket + index **first** (cell 1), then create the Knowledge Base that uses them (cell 2).
This is the fix for the "index could not be found" error.

In [8]:
# WHAT THIS CELL DOES (plain English):
# - Creates the "vector bucket" and the "index" that will store the document embeddings
#   (the numeric form of the text that lets us search by meaning).
# - dimension=1024 and cosine distance must MATCH the Titan embedding model we use.
# - The try/except blocks mean: if the bucket or index already exists, just carry on.
# - At the end it prints the ARNs (unique IDs) we'll pass to the Knowledge Base next.
import boto3
import lab_helpers.utils as u

account = u.get_aws_account_id()
s3v = boto3.client("s3vectors", region_name=u.REGION)

vector_bucket_name = u.KB_NAME                # e.g. careconnect-kb-sdk
index_name = f"{u.KB_NAME}-index"

# 1) Create the vector bucket (idempotent-ish: ignore if it already exists)
try:
    s3v.create_vector_bucket(vectorBucketName=vector_bucket_name)
    print("Created vector bucket:", vector_bucket_name)
except Exception as e:
    print("vector bucket note:", e)

# 2) Create the index. Dimension MUST match the embedding model:
#    Titan Text Embeddings V2 = 1024 dims, cosine distance.
try:
    s3v.create_index(
        vectorBucketName=vector_bucket_name,
        indexName=index_name,
        dataType="float32",
        dimension=1024,
        distanceMetric="cosine",
    )
    print("Created index:", index_name)
except Exception as e:
    print("index note:", e)

# 3) Resolve the ARNs we'll hand to Bedrock
vb = s3v.get_vector_bucket(vectorBucketName=vector_bucket_name)["vectorBucket"]
vector_bucket_arn = vb["vectorBucketArn"]
index_arn = f"arn:aws:s3vectors:{u.REGION}:{account}:bucket/{vector_bucket_name}/index/{index_name}"
print("vector_bucket_arn:", vector_bucket_arn)
print("index_arn:", index_arn)

Created vector bucket: careconnect-kb-sdk
Created index: careconnect-kb-sdk-index
vector_bucket_arn: arn:aws:s3vectors:us-east-1:831963379350:bucket/careconnect-kb-sdk
index_arn: arn:aws:s3vectors:us-east-1:831963379350:bucket/careconnect-kb-sdk/index/careconnect-kb-sdk-index


In [9]:
# WHAT THIS CELL DOES (plain English):
# - Creates the Bedrock Knowledge Base itself and connects it to the vector store above.
# - 'embeddingModelArn' says which model turns text into searchable numbers (Titan v2).
# - If creation fails, it prints the exact error so you can adjust (schemas change between
#   library versions). On success it saves nothing yet — the next cell waits for it to be ready.
bedrock_agent = boto3.client("bedrock-agent", region_name=u.REGION)
account = u.get_aws_account_id()

embed_arn = f"arn:aws:bedrock:{u.REGION}::foundation-model/{u.EMBED_MODEL_ID}"

# NOTE: S3 Vectors KB creation via SDK — adjust to your botocore version if needed.
try:
    kb = bedrock_agent.create_knowledge_base(
        name=u.KB_NAME,
        description="CareConnect approved Riverside Health documents (SDK build).",
        roleArn=kb_role_arn,
        knowledgeBaseConfiguration={
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {"embeddingModelArn": embed_arn},
        },
        storageConfiguration={
            "type": "S3_VECTORS",
            "s3VectorsConfiguration": {
                # Let Bedrock create/manage the vector bucket + index for the lab.
                "vectorBucketArn": f"arn:aws:s3vectors:{u.REGION}:{account}:bucket/{u.KB_NAME}",
                "indexName": f"{u.KB_NAME}-index",
            },
        },
    )
    kb_id = kb["knowledgeBase"]["knowledgeBaseId"]
    print("Created KB:", kb_id)
except Exception as e:
    print("create_knowledge_base failed — inspect and adjust schema:\n", e)
    raise

Created KB: ZI0R8MDWXM


In [10]:
# WHAT THIS CELL DOES (plain English):
# - Waits until the Knowledge Base finishes being created and becomes 'ACTIVE'.
# - Then saves its ID into Parameter Store so every later lab can find it without hard-coding.
u.wait_for_kb_ready(kb_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/kb_id", kb_id)
print("Saved KB id to SSM:", kb_id)

KB status: ACTIVE
Saved KB id to SSM: ZI0R8MDWXM


### Step 6 — Connect the documents and index them ("sync")

**What:** attach the approved-documents bucket to the Knowledge Base as a **data source**,
then start an **ingestion job** that reads the documents, splits them up, converts them to
embeddings, and stores them in the index.

**Why:** creating the Knowledge Base is like building an empty library; this step actually
puts the books on the shelves. Until this "sync" completes, searches return nothing.

In [11]:
# WHAT THIS CELL DOES (plain English):
# - Tells the Knowledge Base WHERE its documents live (our S3 bucket + 'approved/' folder).
# - Then starts an "ingestion job": Bedrock reads each document, chunks it, turns the chunks
#   into embeddings, and loads them into the index so they become searchable.
ds_resp = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name=u.name("careconnect-approved-documents"),
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{u.EXISTING_DOCS_BUCKET}",
            "inclusionPrefixes": [u.DOCS_PREFIX],
        },
    },
)
data_source_id = ds_resp["dataSource"]["dataSourceId"]
print("Data source:", data_source_id)

job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id, dataSourceId=data_source_id)
print("Ingestion job started:", job["ingestionJob"]["ingestionJobId"])

Data source: G0XEP8RONJ
Ingestion job started: NESITLPOZB


In [12]:
# WHAT THIS CELL DOES (plain English):
# - Repeatedly checks the ingestion job's status every 15 seconds until it says COMPLETE
#   (or FAILED). This is just waiting for the indexing to finish.
# Poll the ingestion job to completion
while True:
    j = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=data_source_id,
        ingestionJobId=job["ingestionJob"]["ingestionJobId"])["ingestionJob"]
    print("ingestion:", j["status"])
    if j["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)

ingestion: IN_PROGRESS
ingestion: COMPLETE


### Step 7 — Test that search works

**What:** ask the Knowledge Base a sample question and print the matching document passages
and their relevance scores.

**Why:** this is a quick sanity check that the whole pipeline (documents → embeddings →
search) actually works before we build agents on top of it. If this returns relevant
passages, the foundation is solid.

In [13]:
# WHAT THIS CELL DOES (plain English):
# - Sends a test question to the Knowledge Base and asks for the 3 best-matching passages.
# - Prints each passage's source file and a relevance score (higher = better match).
# - This proves search is working end to end.
rt = boto3.client("bedrock-agent-runtime", region_name=u.REGION)
r = rt.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "How should a patient prepare for a colonoscopy?"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
for res in r["retrievalResults"]:
    print(res["location"].get("s3Location", {}).get("uri"), "score", res.get("score"))

s3://careconnect-approved-docs/approved/procedure-prep-endoscopy.md score 0.7278532385826111
s3://careconnect-approved-docs/approved/procedure-prep-ct-contrast.md score 0.6811266243457794
s3://careconnect-approved-docs/approved/appointment-preparation-general.md score 0.6560361385345459


### Step 8 — Create the Safety Guardrail (the safety net)

**What:** create a Bedrock **Guardrail** that (a) blocks four medical topic areas —
Diagnosis, Dosage, Treatment, Triage — and (b) masks personal information (email, phone,
name, address, and Riverside's `MRN-########` medical record numbers).

**Why:** this is the core safety promise of CareConnect. Even if the model were somehow
prompted to give medical advice, the Guardrail intercepts it. And masking personal data
protects patient privacy. We save the Guardrail's ID so every later component can apply it.

**The use case:** a patient asks "should I increase my insulin?" — the Guardrail ensures the
system refuses and points them to a clinician, rather than answering.

In [14]:
# WHAT THIS CELL DOES (plain English):
# - Creates the safety Guardrail with:
#     * "denied topics" — Diagnosis, Dosage, Treatment, Triage — that must never be answered.
#     * PII masking — emails, phones, names, addresses, and MRN numbers get hidden.
# - 'blockedInputMessaging'/'blockedOutputsMessaging' is the polite refusal message shown
#   when someone asks a medical question.
# - It then creates a versioned copy of the Guardrail and saves the ID + version into
#   Parameter Store for the rest of the project to use.
bedrock = boto3.client("bedrock", region_name=u.REGION)

blocked_msg = ("I can help with approved Riverside Health information, but I cannot "
               "provide medical diagnosis, medication dosage changes, treatment "
               "recommendations, or medical triage. Please contact a qualified "
               "healthcare professional for medical advice.")

g = bedrock.create_guardrail(
    name=u.GUARDRAIL_NAME,
    description="Safety and privacy guardrail for CareConnect (SDK build).",
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
    topicPolicyConfig={"topicsConfig": [
        {"name": "Medical Diagnosis", "type": "DENY",
         "definition": "Questions or statements that identify, confirm, or infer a medical condition based on symptoms, test results, or patient information."},
        {"name": "Medication Dosage", "type": "DENY",
         "definition": "Questions, guidance, or recommendations about medication dose, changing a dose, medication frequency, or taking more or less medication than prescribed."},
        {"name": "Treatment Recommendation", "type": "DENY",
         "definition": "Questions or recommendations that choose, prescribe, or recommend a medical treatment, medication, procedure, or therapy for a patient."},
        {"name": "Medical Triage", "type": "DENY",
         "definition": "Questions that decide how urgently a patient needs care or which level of care to seek."},
    ]},
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "NAME", "action": "ANONYMIZE"},
            {"type": "ADDRESS", "action": "ANONYMIZE"},
        ],
        "regexesConfig": [
            {"name": "MRN", "pattern": "MRN-[0-9]{8}", "action": "ANONYMIZE",
             "description": "Synthetic Riverside Health Medical Record Number."},
        ],
    },
)
guardrail_id = g["guardrailId"]
print("Guardrail:", guardrail_id)

ver = bedrock.create_guardrail_version(guardrailIdentifier=guardrail_id,
                                       description="v1 for CareConnect SDK build")
guardrail_version = ver["version"]
u.put_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_id", guardrail_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_version", guardrail_version)
print("Guardrail version:", guardrail_version)

Guardrail: q0c8weyvyrxi
Guardrail version: 1


## Lab 0 complete ✅

You now have, all suffixed `-sdk` and recorded in SSM:
- a Knowledge Base over the **reused** approved-docs bucket, synced and tested
- the Patient Safety Guardrail (versioned)

Next: **lab-01** builds the Retrieval and Document-Processing agents.